## Baseline Evaluation
This is to benchmarking baseline performance. 
This should include following sections:
1. off-shelf LLMs on MedQA 


In [1]:
import torch 
import transformers
import datasets
import json 
import pandas as pd
import os 
from openai import OpenAI
from dotenv import load_dotenv

/vol/bitbucket/hl2622/fyp/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = datasets.load_dataset("bigbio/med_qa", "med_qa_en_source", trust_remote_code=True)
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['meta_info', 'question', 'answer_idx', 'answer', 'options'],
        num_rows: 10178
    })
    test: Dataset({
        features: ['meta_info', 'question', 'answer_idx', 'answer', 'options'],
        num_rows: 1273
    })
    validation: Dataset({
        features: ['meta_info', 'question', 'answer_idx', 'answer', 'options'],
        num_rows: 1272
    })
})
{'meta_info': 'step2&3', 'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is 

In [12]:
# format dataset questons 
def format_question(sample):
    print(sample)
    options = "\n".join(
        [f"{key}.{val}" for key, val in sample["options"]]
    )

    prompts = (
        f"You are a medical student. Answer the following question:\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{options}\n\n"
        f"Answer with only the letter of the correct option (A, B, C, or D)."
    )

    return prompts 

# parse answer 
def parse_answer(ans):
    parsed_ans = ans.strip().upper()
    if parsed_ans in ["A", "B", "C", "D"]:
        return parsed_ans
    return "UNKNOWN"

# evaluate model 
def evaluate_model(model_fn, questions, model_name="model"):
    total = len(questions)
    correct = 0 
    res = []

    for i, sample in enumerate(questions):
        prompt = format_question(sample)
        model_answer = model_fn(prompt)
        parsed_answer = parse_answer(model_answer)
        ground_truth = sample["answer_idx"]
        is_correct = parsed_answer == ground_truth
        if is_correct:
            correct += 1

        results.append({
            "id": i, 
            "question": sample["question"],
            "ground_truth": ground_truth,
            "model_answer": parsed_answer,
            "is_correct": is_correct
        })

    accuracy = correct / total if total > 0 else 0
    print(f"{model_name} Accuracy: {accuracy:.2%}")
    return results, accuracy

In [14]:
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("GPU memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

GPU available: True
GPU name: NVIDIA GeForce RTX 4080
GPU memory: 16.718954496 GB


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1,
    temperature=0.2,
    device_map="auto",
    do_sample=False
)

def local_llama_fn(prompt):
    messages = [{"role": "user", "content": prompt}]

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return pipe(formatted)[0]["generated_text"][len(prompt):].strip()

test_ds = list(dataset["test"])[:200]

llama3_results, llama3_accuracy = evaluate_model(local_llama_fn, test_ds, model_name=model_id)

print(f"Llama3 Results: {gpt4o_results[:5]}; Accuracy: {gpt4o_accuracy:.2%}")


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
401 Client Error. (Request ID: Root=1-69c461bb-63adc69a759382b71f42f6b7;7f2137b0-6ed0-4e31-a94b-7ccf8002ed6c)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.